### ***LOADING THE WINDOWED AUDIO***

In [1]:
from pathlib import Path

SAMPLE_RATE: int = 16_000
N_MELS: int = 26
N_MFCC: int = 13
F_MIN: float = 0.0
F_MAX: float = SAMPLE_RATE / 2
ROLLOFF_PERCENT: float = 0.85

FRAME_SIZE: int = 400
N_FFT: int = FRAME_SIZE

BATCH_SIZE: int = 4_096
# arbitrary btw
# batch size is just to save on memory and space
# not necessary but Little Ms. LLM gave it so ok

NOTEBOOK_DIR: Path = Path.cwd()
DIR_REPOSITORY_ROOT: Path = NOTEBOOK_DIR
DIR_INPUT_FFT: Path = DIR_REPOSITORY_ROOT / "preprocessing-03-fft.npy"
DIR_OUTPUT_MFCC: Path = DIR_REPOSITORY_ROOT / "extracted-mfcc.npz"

# check if file eixists
if not DIR_INPUT_FFT.is_file():
    raise FileNotFoundError(
        f"Missing FFT data: {DIR_INPUT_FFT.resolve()}\n"
        "Run preprocessing-01-framing.ipynb, preprocessing-02-windowing.ipynb, "
        "and preprocessing-03-fft.ipynb first."
    )

In [2]:
import numpy as np

INPUT_FFT:np.ndarray = np.load(DIR_INPUT_FFT.resolve(), allow_pickle=True)

### ***CHECKING THE WINDOWED FRAMES***

note : Only load trusted NumPy files with `allow_pickle=True`. A pickled object array can contain unsafe code.

In [3]:
# checking the file contents
if INPUT_FFT.ndim != 2 or INPUT_FFT.shape[1] != 5:
    raise ValueError(f"Expected an FFT array with shape (number_of_frames, 5), got {INPUT_FFT.shape}.")
if len(INPUT_FFT) == 0:
    raise ValueError("The FFT input contains no frames.")
EXPECTED_BINS: int = N_FFT // 2 + 1
first_power = np.asarray(INPUT_FFT[0, 4])
if first_power.shape != (EXPECTED_BINS,):
    raise ValueError(
        f"Expected {EXPECTED_BINS} power bins for N_FFT={N_FFT}, got {first_power.shape}."
    )

n = 25
print("="*50, "SHAPE", f"\t{INPUT_FFT.shape=}", sep="\n")
print("SHAPE", f"\t{INPUT_FFT[n].shape=}", sep="\n")
print("="*50, "DIR", f"\t{INPUT_FFT[n][0]=}", sep="\n")
print("="*50, "FRAME#", f"\t{INPUT_FFT[n][1]=}", sep="\n")

print("="*50, "SHAPE", f"\t{INPUT_FFT[n][2].shape=}", sep="\n")
print("FRAME_DATA", f"\t{INPUT_FFT[n][2]=}", sep="\n")

SHAPE
	INPUT_FFT.shape=(1099018, 5)
SHAPE
	INPUT_FFT[n].shape=(5,)
DIR
	INPUT_FFT[n][0]='/home/jules/files-thesis/tutorial-audio_feature_extraction/Artists/Rapper/Dax-Dr-Dre-Ft-Eminem-Forgot-About-Dre-Remix-(TrendyBeatz.com).mp3'
FRAME#
	INPUT_FFT[n][1]=26
SHAPE
	INPUT_FFT[n][2].shape=(201,)
FRAME_DATA
	INPUT_FFT[n][2]=array([-1.96880591e-03+0.00000000e+00j,  5.74778160e-03-2.61559244e-03j,
       -1.29538476e-02+4.13283153e-04j,  9.10085067e-03+2.97192339e-04j,
       -3.53253068e-04+1.76584511e-03j, -3.59959085e-03-1.27206184e-03j,
        4.90317307e-03+1.46949652e-03j, -4.19604639e-03-2.35905242e-03j,
        1.38837774e-03+9.74024995e-04j, -4.94231936e-04+8.67572671e-04j,
        7.66270212e-04-4.37131501e-04j, -1.21005450e-03+1.13425532e-03j,
        1.36547966e-03-3.05114983e-04j, -1.24149700e-03-7.72156636e-04j,
       -7.13761256e-04+5.30444202e-04j,  6.44015614e-04+1.71321357e-04j,
        3.24108260e-04-1.50657620e-03j, -5.46073308e-04+1.38490007e-03j,
        3.36712168e-04

### ***MEL FILTERBANK*** and helper function

In [4]:
import librosa

MEL_FILTERBANK: np.ndarray = librosa.filters.mel(
    sr=SAMPLE_RATE,
    n_fft=N_FFT,
    n_mels=N_MELS,
    fmin=F_MIN,
    fmax=F_MAX,
    htk=True,
    norm=None,
    dtype=np.float32,
)

assert MEL_FILTERBANK.shape == (N_MELS, EXPECTED_BINS)
assert MEL_FILTERBANK.dtype == np.float32
assert np.isfinite(MEL_FILTERBANK).all()
assert (MEL_FILTERBANK >= 0).all()

print(f"Mel filterbank shape: {MEL_FILTERBANK.shape}")

Mel filterbank shape: (26, 201)


In [5]:
from scipy.fft import dct

LOG_FLOOR: float = np.finfo(np.float32).eps

def power_to_mfcc(power_spectrum: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    power = np.asarray(power_spectrum, dtype=np.float32)

    # checking data shape
    if power.shape != (EXPECTED_BINS,):
        raise ValueError(f"Expected power shape {(EXPECTED_BINS,)}, got {power.shape}.")
    if not np.isfinite(power).all() or (power < 0).any():
        raise ValueError("The power spectrum must contain finite, nonnegative values.")

    # uses the mel filterbank
    mel_energies = MEL_FILTERBANK @ power

    # np.maximum filters anything smallter than LOG_FLOOR for noise
    # np.log gets the log of the mel_energies
    log_mel_energies:np.ndarray = np.log(np.maximum(mel_energies, LOG_FLOOR))

    # the DCT(Discrete Cosine Transform) step
    coefficients:np.ndarray = dct(log_mel_energies, type=2, norm="ortho")[:N_MFCC]
    return coefficients.astype(np.float32), log_mel_energies.astype(np.float32)

### ***EXTRACT THE DATA***

In [6]:
MFCC_FRAMES: np.ndarray = np.empty((len(INPUT_FFT), 3), dtype=object)
LOG_MEL_ENERGIES: np.ndarray = np.empty((len(INPUT_FFT), N_MELS), dtype=np.float32)

for row_index, row in enumerate(INPUT_FFT):
    source_file, frame_number, _, _, power_spectrum = row
    coefficients, log_mel_energies = power_to_mfcc(power_spectrum)
    MFCC_FRAMES[row_index] = [source_file, frame_number, coefficients]
    LOG_MEL_ENERGIES[row_index] = log_mel_energies

print(f"Extracted {N_MFCC} MFCCs from each of {len(MFCC_FRAMES)} frames")
print("Columns: source_file, frame_number, mfcc")

Extracted 13 MFCCs from each of 1099018 frames
Columns: source_file, frame_number, mfcc


### ***SAVE THE MFCC DATA***

This notebook saves an `.npz` file, not the single object `.npy` file used by the earlier preprocessing steps.

`MFCC_FRAMES` uses `dtype=object`. Each row holds a path, a frame number, and a float32 vector. NumPy cannot memory-map object arrays. A check cell would have to load the full file. Separate arrays fix that.

The paths, frame numbers, and MFCC vectors go into different members. The check cell can map the file and read only the start and the end.

Use `np.savez` instead of `np.savez_compressed`. Compressed members cannot be memory-mapped.

In [7]:
MAX_PATH_BYTES: int = max(len(str(row[0]).encode("utf-8")) for row in MFCC_FRAMES)

SOURCE_FILES: np.ndarray = np.array(
    [str(row[0]) for row in MFCC_FRAMES],
    dtype=f"S{MAX_PATH_BYTES}",)
FRAME_NUMBERS: np.ndarray = np.array(
    [row[1] for row in MFCC_FRAMES], 
    dtype=np.int32)
MFCC_VALUES: np.ndarray = np.array(
    [row[2] for row in MFCC_FRAMES], 
    dtype=np.float32
)

DIR_OUTPUT_MFCC.parent.mkdir(parents=True, exist_ok=True)
np.savez(
    DIR_OUTPUT_MFCC,
    source_files=SOURCE_FILES,
    frame_numbers=FRAME_NUMBERS,
    mfcc=MFCC_VALUES,
)

print(f"{MAX_PATH_BYTES=}")
print(f"Saved {len(MFCC_VALUES)} frames with shape {MFCC_VALUES.shape}")
print(f"Output: {DIR_OUTPUT_MFCC.resolve()}")
print("Members: source_files (bytes), frame_numbers (int64), mfcc (float32)")

MAX_PATH_BYTES=142
Saved 1099018 frames with shape (1099018, 13)
Output: /home/jules/files-thesis/tutorial-audio_feature_extraction/extracted-mfcc.npz
Members: source_files (bytes), frame_numbers (int64), mfcc (float32)


### ***CHECK THE SAVED DATA***

`mmap_mode="r"` maps the file. The check reads only the first and last rows. It compares the saved values with the arrays still in memory. This check does not examine the middle rows. To check every row, use a full comparison instead.

In [8]:
HEAD_ROWS: int = 5
TAIL_ROWS: int = 5

with np.load(DIR_OUTPUT_MFCC, mmap_mode="r") as saved:
    source_files = saved["source_files"]
    frame_numbers = saved["frame_numbers"]
    mfcc_values = saved["mfcc"]

    for rows in (slice(0, HEAD_ROWS), slice(-TAIL_ROWS, None)):
        expected_sources = np.asarray([str(row[0]) for row in MFCC_FRAMES[rows]], dtype=str)
        expected_frames = np.asarray([row[1] for row in MFCC_FRAMES[rows]], dtype=np.int32)
        expected_mfcc = np.array([row[2] for row in MFCC_FRAMES[rows]], dtype=np.float32)

        assert np.array_equal(source_files[rows].astype(str), expected_sources)
        assert np.array_equal(frame_numbers[rows], expected_frames)
        assert mfcc_values[rows].shape == expected_mfcc.shape
        assert mfcc_values[rows].dtype == np.float32
        assert np.isfinite(mfcc_values[rows]).all()
        assert np.array_equal(mfcc_values[rows], expected_mfcc)

print(f"Integrity checked: first {HEAD_ROWS} and last {TAIL_ROWS} saved frames match memory.")

Integrity checked: first 5 and last 5 saved frames match memory.


In [9]:
import csv

DIR_OUTPUT_MFCC_CSV: Path = DIR_OUTPUT_MFCC.with_suffix(".csv")
mfcc_columns: list[str] = [
    f"mfcc{coefficient:02}" for coefficient in range(1, N_MFCC + 1)
]

with DIR_OUTPUT_MFCC_CSV.open(
    mode="w",
    newline="",
    encoding="utf-8",
) as csv_file:
    writer = csv.writer(csv_file)
    writer.writerow(["source_file", "frame_number", *mfcc_columns])

    for source_file, frame_number, coefficients in MFCC_FRAMES:
        coefficients32 = np.asarray(coefficients, dtype=np.float32)

        writer.writerow(
            [
                str(source_file),
                int(frame_number),
                *(format(float(value), ".9g") for value in coefficients32),
            ]
        )

print(f"CSV output: {DIR_OUTPUT_MFCC_CSV.resolve()}")

CSV output: /home/jules/files-thesis/tutorial-audio_feature_extraction/extracted-mfcc.csv
